In [0]:
from pyspark.sql.functions import * 
from pyspark.sql.types import *
from pyspark.sql.window import Window
from pyspark.sql import functions as F

In [0]:
df = spark.read.table('fintech360.schema_validation.upi_data')

In [0]:
df.display()

In [0]:
df = df.withColumn("transaction_ts", unix_timestamp(col("transaction_ts"))) 

In [0]:
df = df.withColumn(
    "is_refund",
    when(col("is_refund") == True, 1).otherwise(0)
)

display(df)

In [0]:
df = df.withColumn(
    "payer_account_type",
    regexp_extract(
        col("upi_details_json"),
        r'payer_account_type"":""([^"]+)',
        1
    )
)

display(df.select("upi_details_json", "payer_account_type"))

In [0]:
df = df.withColumn(
    "transaction_status",
    when(col("transaction_status") == "SUCCESS", 1).otherwise(0)
)

display(df)

In [0]:
from pyspark.sql import functions as F

df = (
    df
    .withColumn("created_at", F.col("created_at").cast("long") * 1000)
    .withColumn("updated_at", F.col("updated_at").cast("long") * 1000)
)

df.select("created_at", "updated_at").show(10, truncate=False)

In [0]:
df = df.withColumn("_ingested_at", unix_timestamp(col("_ingested_at").cast("timestamp")))
df.limit(5).show()

In [0]:
(df.write
.format('delta')
.mode('append')
.saveAsTable('fintech360.silver.upi_data'))